In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install opencv-python-headless scikit-image numpy pandas xgboost

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from skimage.feature import hog
import tensorflow as tf
from sklearn.model_selection import train_test_split
import shutil
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

In [ ]:
base_dir = "/content/drive/MyDrive/Data"

cat_data = [os.path.join(base_dir, "Cat", f) for f in os.listdir(os.path.join(base_dir, "Cat")) if f.endswith('.jpg')]
dog_data = [os.path.join(base_dir, "Dog", f) for f in os.listdir(os.path.join(base_dir, "Dog")) if f.endswith('.jpg')]



In [ ]:
IMG_SIZE = 128

def load_data(data_paths):
    features = []
    labels = []
    for filepath in data_paths:
        img = cv2.imread(filepath)
        if img is None:
            continue
        # Resize và chuyển sang grayscale
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Extract HOG features (không visualize)
        hog_features = hog(
            gray,
            orientations=8,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            visualize=False
        )
        features.append(hog_features)
        # Nhãn: dog=1, cat=0
        labels.append(1 if 'Dog' in filepath else 0)

    return np.array(features), np.array(labels)

# Load dữ liệu
X,y = load_data(cat_data + dog_data)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (880, 7200)
y_train shape: (880,)
X_test shape: (221, 7200)
y_test shape: (221,)


In [ ]:
svm = SVC(kernel='rbf', C=3)
svm.fit(X_train, y_train)

SVC(C=3)

In [ ]:
# Predict
y_pred = svm.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 72.85%
